In [ ]:
import torch
import numpy as np
import time
import os
from transformers import (
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score
import gc # 가비지 컬렉터 (메모리 정리용)

In [ ]:
# --- (단계 1) 환경 설정 ---
# GPU 사용 여부 확인 및 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 모델 ID 및 설정
MODEL_ID = "klue/roberta-base"
OUTPUT_DIR = "./results_ynat"
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
BATCH_SIZE = 8

# 출력 디렉토리 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GPU 메모리 사용량 측정을 위한 함수
def get_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024**3)
    return 0.0

# 파라미터 개수 계산 함수
def count_parameters(model, trainable_only=True):
    """모델의 파라미터 개수를 계산"""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    else:
        return sum(p.numel() for p in model.parameters())

In [ ]:
# --- (단계 2) 데이터셋 로드 및 전처리 ---

# 데이터셋 로드 (기존 노트북 코드 활용)
print("\n## 1. 데이터셋 로드 및 전처리")
klue_tc_train = load_dataset('klue', 'ynat', split='train')
klue_tc_eval = load_dataset('klue', 'ynat', split='validation')

# 불필요한 컬럼 제거
klue_tc_train = klue_tc_train.remove_columns(['guid', 'url', 'date'])
klue_tc_eval = klue_tc_eval.remove_columns(['guid', 'url', 'date'])

# 카테고리 정보
klue_tc_label_names = klue_tc_train.features['label'].names
NUM_LABELS = len(klue_tc_label_names)
print(f"Number of Labels: {NUM_LABELS}, Names: {klue_tc_label_names}")

# 학습/검증/테스트 데이터셋 분할
# train_dataset (학습용): 10000개
train_dataset = klue_tc_train.train_test_split(test_size=10000, shuffle=True, seed=42)['test']
# eval_data (검증/테스트 분할용)
eval_data = klue_tc_eval.train_test_split(test_size=2000, shuffle=True, seed=42)
# test_dataset (최종 평가용): 2000개
test_dataset = eval_data['test']
# valid_dataset (학습 중 검증용): 7107개 중 1000개
valid_dataset = eval_data['train'].train_test_split(test_size=1000, shuffle=True, seed=42)['test']

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(valid_dataset)}, Test samples: {len(test_dataset)}")


In [ ]:

# 토크나이저 로드 및 토큰화 함수
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# torch 포맷으로 변환 및 불필요한 컬럼 제거
train_tokenized = train_tokenized.remove_columns(['title']).with_format("torch")
valid_tokenized = valid_tokenized.remove_columns(['title']).with_format("torch")
test_tokenized = test_tokenized.remove_columns(['title']).with_format("torch")
test_tokenized = test_tokenized.rename_column("label", "labels")
valid_tokenized = valid_tokenized.rename_column("label", "labels")
train_tokenized = train_tokenized.rename_column("label", "labels")


# 평가 메트릭 함수 정의
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# 데이터 콜레이터 (배치로 묶어주는 역할)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 학습 결과 저장용 딕셔너리
results = {}


In [ ]:


# --- (단계 3) 일반 Fine-Tuning ---
print("\n## 2. 일반 Fine-Tuning 시작")

# 메모리 초기화
if device.type == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_memory = get_gpu_memory()

# 모델 로드
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS
).to(device) # device 지정

# 파라미터 개수
full_total_params = count_parameters(full_model, trainable_only=False)
full_trainable_params = count_parameters(full_model, trainable_only=True)
print(f"Full Model Total Parameters: **{full_total_params:,}**")

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/full_ft",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    logging_dir=f"{OUTPUT_DIR}/full_ft/logs",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LEARNING_RATE,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True if device.type == 'cuda' else False, # GPU 환경에서 혼합 정밀도 사용
)
# Trainer 설정
full_trainer = Trainer(
    model=full_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습 시작 및 시간 측정
start_time = time.time()
full_trainer.train()
end_time = time.time()
training_time_full = end_time - start_time

# 최종 평가 및 결과 기록
full_eval_results = full_trainer.evaluate(test_tokenized)

# 파일 로드 대신 메모리의 모델 객체에서 직접 파라미터 수를 얻음
full_total_params = count_parameters(full_model, trainable_only=False)
full_trainable_params = count_parameters(full_model, trainable_only=True)
print(f"Full Model Trainable Parameters: **{full_trainable_params:,}**")

if device.type == 'cuda':
    peak_memory_full = get_gpu_memory() - start_memory
else:
    peak_memory_full = 0.0

results['Full Fine-Tuning'] = {
    'Accuracy': full_eval_results['eval_accuracy'],
    'Time (s)': training_time_full,
    'Peak GPU Memory (GB)': peak_memory_full,
    'Total Parameters': full_total_params,
    'Trainable Parameters': full_trainable_params, # Full Fine-Tuning의 학습 파라미터
}

# 메모리 정리
del full_model
del full_trainer
if device.type == 'cuda':
    torch.cuda.empty_cache()
gc.collect()
# 생성된 체크포인트 폴더 삭제
import shutil
shutil.rmtree(f"{OUTPUT_DIR}/full_ft", ignore_errors=True)

In [ ]:


# --- (단계 4) LoRA Fine-Tuning ---
print("\n## 3. LoRA Fine-Tuning 시작")

# 메모리 초기화
if device.type == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_memory = get_gpu_memory()

# 기본 모델 로드
lora_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS
).to(device) # device 지정

# LoRA 설정
lora_config = LoraConfig(
    r=16, # LoRA 랭크 (rank)
    lora_alpha=32, # LoRA 스케일링 계수
    target_modules=["query", "key", "value"], # LoRA를 적용할 레이어
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS" # 시퀀스 분류 작업 지정
)

# LoRA 모델로 변환
lora_model = get_peft_model(lora_base_model, lora_config)
print("LoRA Model Summary:")
lora_model.print_trainable_parameters()

lora_trainable_params = count_parameters(lora_model, trainable_only=True)
lora_total_params = count_parameters(lora_model, trainable_only=False)

# TrainingArguments 설정
training_args_lora = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/lora_ft",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    logging_dir=f"{OUTPUT_DIR}/lora_ft/logs",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LEARNING_RATE,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True if device.type == 'cuda' else False, # GPU 환경에서 혼합 정밀도 사용
)

# Trainer 설정
lora_trainer = Trainer(
    model=lora_model,
    args=training_args_lora,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습 시작 및 시간 측정
start_time = time.time()
lora_trainer.train()
end_time = time.time()
training_time_lora = end_time - start_time

# 최종 평가 및 결과 기록
lora_eval_results = lora_trainer.evaluate(test_tokenized)

if device.type == 'cuda':
    peak_memory_lora = get_gpu_memory() - start_memory
else:
    peak_memory_lora = 0.0

results['LoRA Fine-Tuning'] = {
    'Accuracy': lora_eval_results['eval_accuracy'],
    'Time (s)': training_time_lora,
    'Peak GPU Memory (GB)': peak_memory_lora,
    'Total Parameters': lora_total_params,
    'Trainable Parameters': lora_trainable_params, # LoRA의 학습 파라미터
}
# 메모리 정리
del lora_model
del lora_base_model
del lora_trainer
if device.type == 'cuda':
    torch.cuda.empty_cache()
gc.collect()
# 생성된 체크포인트 폴더 삭제
shutil.rmtree(f"{OUTPUT_DIR}/lora_ft", ignore_errors=True)


In [ ]:
# --- (단계 5) 결과 비교 및 분석 ---
print("\n## 4. 최종 결과 비교 및 분석 📊")

# 결과 비교 테이블 출력
print("--- 결과 비교 테이블 ---")
print(f"{'항목':<35} {'일반 Fine-Tuning':<25} {'LoRA Fine-Tuning':<20}")
print("-" * 80)

# ⭐ 수정된 부분: f-string 내에서 산술 연산을 제거하고, 전체 문자열 너비를 고정합니다.

# Accuracy
full_acc_str = f"{results['Full Fine-Tuning']['Accuracy'] * 100:.2f}%"
lora_acc_str = f"{results['LoRA Fine-Tuning']['Accuracy'] * 100:.2f}%"
print(f"{'Test Accuracy':<35} {full_acc_str:<25} {lora_acc_str:<20}")

# Training Time
full_time_str = f"{results['Full Fine-Tuning']['Time (s)']/60:.2f} min ({results['Full Fine-Tuning']['Time (s)']:.0f} s)"
lora_time_str = f"{results['LoRA Fine-Tuning']['Time (s)']/60:.2f} min ({results['LoRA Fine-Tuning']['Time (s)']:.0f} s)"
print(f"{'Training Time':<35} {full_time_str:<25} {lora_time_str:<20}")

# Peak GPU Memory Usage
full_mem_str = f"{results['Full Fine-Tuning']['Peak GPU Memory (GB)']:.2f} GB"
lora_mem_str = f"{results['LoRA Fine-Tuning']['Peak GPU Memory (GB)']:.2f} GB"
print(f"{'Peak GPU Memory Usage (GB)':<35} {full_mem_str:<25} {lora_mem_str:<20}")

# Trainable Parameters
full_trainable_str = f"{results['Full Fine-Tuning']['Trainable Parameters']:,.0f}"
lora_trainable_str = f"{results['LoRA Fine-Tuning']['Trainable Parameters']:,.0f}"
print(f"{'Trainable Parameters':<35} {full_trainable_str:<25} {lora_trainable_str:<20}")

# Total Parameters
full_total_str = f"{results['Full Fine-Tuning']['Total Parameters']:,.0f}"
lora_total_str = f"{results['LoRA Fine-Tuning']['Total Parameters']:,.0f}"
print(f"{'Total Parameters':<35} {full_total_str:<25} {lora_total_str:<20}")
print("-" * 80)

# 분석 요약
print("\n### 📝 결과 분석 요약")
# 정확도 비교
accuracy_diff = results['LoRA Fine-Tuning']['Accuracy'] - results['Full Fine-Tuning']['Accuracy']
accuracy_comment = "유사한 수준입니다." if abs(accuracy_diff) < 0.01 else ("더 높습니다." if accuracy_diff > 0 else "더 낮습니다.")
print(f"- **정확도:** LoRA Fine-Tuning({results['LoRA Fine-Tuning']['Accuracy'] * 100:.2f}%)의 성능은 일반 Fine-Tuning({results['Full Fine-Tuning']['Accuracy'] * 100:.2f}%)과 **{accuracy_comment}**")

# 학습 시간 비교
time_ratio = results['Full Fine-Tuning']['Time (s)'] / results['LoRA Fine-Tuning']['Time (s)']
print(f"- **학습 효율성 (시간):** LoRA 학습 시간이 일반 학습 시간보다 **{time_ratio:.1f} 배** 빠릅니다. (시간 절약)")

# 자원 사용량 비교 (GPU 사용 가능한 경우만 분석)
if device.type == 'cuda' and results['LoRA Fine-Tuning']['Peak GPU Memory (GB)'] > 0:
    memory_ratio = results['Full Fine-Tuning']['Peak GPU Memory (GB)'] / results['LoRA Fine-Tuning']['Peak GPU Memory (GB)']
    print(f"- **자원 사용량 (메모리):** LoRA는 GPU 메모리를 **{memory_ratio:.1f} 배** 적게 사용합니다. (GPU 메모리 절약)")
else:
    print("- **자원 사용량 (메모리):** GPU 환경이 아니거나 측정값이 0이라 비교가 어렵습니다.")

# 모델 크기 비교
param_ratio = results['Full Fine-Tuning']['Trainable Parameters'] / results['LoRA Fine-Tuning']['Trainable Parameters']
print(f"- **모델 저장 용량:** 일반 Fine-Tuning은 전체 모델 가중치 **{results['Full Fine-Tuning']['Trainable Parameters']:,.0f}**개를 학습하는 반면, LoRA는 **{results['LoRA Fine-Tuning']['Trainable Parameters']:,.0f}**개만 학습합니다. 이는 **{param_ratio:.0f} 배** 이상 적은 수치로, 저장 및 배포에 매우 효율적입니다.")